# Percepción de seguridad y victimización — OMSCGR

Notebook de análisis reproducible a partir de la base **EVPSC** (`final.dta`),
Módulo 7 (Percepción de Inseguridad y Confianza Institucional). Calcula todos
los agregados (`APP_DATA`) que alimentan el HTML del diagnóstico: percepción
de (in)seguridad, victimización completa (robo y agresión/amenaza en 4
lugares cada uno), confianza institucional, percepción de espacios
específicos, exposición a armas, un modelo de riesgo, y el desglose
territorial.

**Cómo usar en Google Colab:**
1. Ejecuta la celda de instalación de dependencias.
2. Sube `final.dta` cuando se te pida.
3. Corre el resto de celdas en orden.
4. Al final se guarda `percepcion_data.json`, descargable desde el panel de archivos.


## 1. Dependencias e insumos

In [ ]:
!pip install statsmodels scikit-learn -q

In [ ]:
from google.colab import files
import os

if not os.path.exists('final.dta'):
    print("Sube el archivo final.dta:")
    uploaded = files.upload()
    for fname in uploaded:
        if fname != 'final.dta':
            os.rename(fname, 'final.dta')
print("Archivo listo:", os.path.exists('final.dta'))

In [ ]:
# Alternativa: montar Google Drive en vez de subir el archivo cada vez
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/ruta/a/final.dta'
DATA_PATH = 'final.dta'

In [ ]:
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

SI, NO = '1. Sí', '2. No'
MAP_SEG = {'1. Muy inseguro/a': 1, '2. Inseguro/a': 2, '3. Seguro/a': 3, '4. Muy seguro/a': 4}
MAP_CONF = {'1. Desconfío bastante': 1, '2. Desconfío': 2, '3. Confío': 3, '4. Confío bastante': 4}

## 2. Cargar la base

`convert_categoricals=False` porque la base ya trae las respuestas como
texto ("1. Sí" / "2. No" / escalas), no como categorías numéricas etiquetadas.

In [ ]:
df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Filas:", len(df), " | Columnas:", df.shape[1])
print("Unidades educativas:", df['ue'].nunique())

mod = df[df['pi5'] != ''].copy()  # subconjunto con el módulo 7
print("Con módulo 7 (percepción/victimización):", len(mod), f"({100*len(mod)/len(df):.1f}%)")
df.head(3)

## 3. Conciliación de la base

El Módulo 7 se aplicó a un subconjunto por diseño modular del cuestionario:
16 escuelas FISCAL y 2 particulares no lo recibieron.

In [ ]:
conciliacion = {
    "n_total": int(len(df)),
    "n_modulo": int(len(mod)),
    "pct_modulo": round(100 * len(mod) / len(df), 1),
    "n_unidades_educativas": int(df['ue'].nunique()),
}
conciliacion

## 4. Funciones auxiliares

`pct_scale` para variables de escala (percepción de seguridad, confianza);
`pct_si` para variables Sí/No (victimización, armas).

In [ ]:
def pct_scale(series, mapping, threshold, direction='le'):
    s = series.map(mapping)
    valid = s.notna()
    n = int(valid.sum())
    if n == 0:
        return np.nan, 0
    if direction == 'le':
        p = 100 * (s[valid] <= threshold).mean()
    else:
        p = 100 * (s[valid] >= threshold).mean()
    return p, n

def pct_si(series):
    valid = series.isin([SI, NO])
    n = int(valid.sum())
    if n == 0:
        return np.nan, 0
    return 100 * (series[valid] == SI).mean(), n

## 5. Percepción de (in)seguridad (PI1-4)

Qué tan seguro se siente el estudiante en 4 lugares: ruta casa-escuela,
dentro de la escuela, alrededores de la escuela, y su barrio.

In [ ]:
labels_pi14 = {'pi1': 'Ruta casa-escuela', 'pi2': 'Dentro de la escuela',
               'pi3': 'Alrededores de la escuela', 'pi4': 'Barrio'}
percepcion = []
for v, lab in labels_pi14.items():
    p, n = pct_scale(mod[v], MAP_SEG, 2, 'le')
    percepcion.append({"var": v, "label": lab, "pct_inseguro": round(p, 1), "n": n})

percepcion.sort(key=lambda r: r['pct_inseguro'], reverse=True)
pd.DataFrame(percepcion)

In [ ]:
import matplotlib.pyplot as plt

perc_df = pd.DataFrame(percepcion).sort_values('pct_inseguro')
plt.figure(figsize=(8, 3))
plt.barh(perc_df['label'], perc_df['pct_inseguro'], color='#1d9ebe')
plt.xlabel('% que se siente inseguro/muy inseguro')
plt.title('Percepción de inseguridad por lugar')
plt.tight_layout()
plt.show()

## 6. Victimización completa (PI5-12)

8 ítems: robo y agresión/amenaza, cada uno medido en 4 lugares (dentro de
la escuela, alrededores, barrio, trayecto).

In [ ]:
labels_pi = {
    'pi5': 'Robo — dentro de la escuela', 'pi6': 'Robo — alrededores de la escuela',
    'pi7': 'Robo — trayecto', 'pi8': 'Robo — barrio',
    'pi9': 'Agresión/amenaza — dentro de la escuela', 'pi10': 'Agresión/amenaza — alrededores',
    'pi11': 'Agresión/amenaza — barrio', 'pi12': 'Agresión/amenaza — trayecto',
}
victimizacion = []
for v, lab in labels_pi.items():
    p, n = pct_si(mod[v])
    victimizacion.append({"var": v, "label": lab, "pct": round(p, 1), "n": n})

victimizacion.sort(key=lambda r: r['pct'], reverse=True)
pd.DataFrame(victimizacion)

In [ ]:
# Resumen: victimización en AL MENOS UNO de los 8 tipos/lugares
valid_all = mod[list(labels_pi.keys())].apply(lambda c: c.isin([SI, NO])).all(axis=1)
any_victim = mod.loc[valid_all, list(labels_pi.keys())].apply(lambda c: c == SI)

victimizacion_resumen = {
    "pct_alguna_vez_cualquiera": round(100 * any_victim.any(axis=1).mean(), 1),
    "pct_robo_algun_lugar": round(100 * any_victim[['pi5', 'pi6', 'pi7', 'pi8']].any(axis=1).mean(), 1),
    "pct_agresion_algun_lugar": round(100 * any_victim[['pi9', 'pi10', 'pi11', 'pi12']].any(axis=1).mean(), 1),
    "n": int(valid_all.sum()),
}
print(f"Víctima alguna vez (cualquiera): {victimizacion_resumen['pct_alguna_vez_cualquiera']}%")
print(f"Por robo en algún lugar: {victimizacion_resumen['pct_robo_algun_lugar']}%")
print(f"Por agresión/amenaza en algún lugar: {victimizacion_resumen['pct_agresion_algun_lugar']}%")
victimizacion_resumen

In [ ]:
vict_df = pd.DataFrame(victimizacion).sort_values('pct')
plt.figure(figsize=(8, 4))
plt.barh(vict_df['label'], vict_df['pct'], color='#f5943d')
plt.xlabel('% víctima alguna vez')
plt.title('Victimización por tipo y lugar')
plt.tight_layout()
plt.show()

## 7. Confianza institucional (CI1-5)

In [ ]:
labels_ci15 = {'ci1': 'Autoridades de la escuela', 'ci2': 'Policía', 'ci3': 'Militares',
               'ci4': 'Bomberos', 'ci5': 'Agentes de Control Metropolitano'}
confianza = []
for v, lab in labels_ci15.items():
    p, n = pct_scale(mod[v], MAP_CONF, 3, 'ge')
    confianza.append({"var": v, "label": lab, "pct_confia": round(p, 1), "n": n})

confianza.sort(key=lambda r: r['pct_confia'], reverse=True)
pd.DataFrame(confianza)

## 8. Percepción de seguridad en espacios específicos (CI6-10)

In [ ]:
labels_ci610 = {'ci6': 'Canchas/áreas recreativas del barrio', 'ci7': 'Canchas de la escuela',
                'ci8': 'Baños de la escuela', 'ci9': 'Aulas de la escuela',
                'ci10': 'Áreas verdes/recreativas de la escuela'}
espacios = []
for v, lab in labels_ci610.items():
    p, n = pct_scale(mod[v], MAP_SEG, 2, 'le')
    espacios.append({"var": v, "label": lab, "pct_inseguro": round(p, 1), "n": n})

espacios.sort(key=lambda r: r['pct_inseguro'], reverse=True)
pd.DataFrame(espacios)

## 9. Exposición a armas (EX9-14)

Ítems no explorados en el diagnóstico de reclutamiento: cuchillos y armas
de fuego, en la escuela y en el barrio.

In [ ]:
labels_armas = {
    'ex10_14': 'Vio cuchillo/navaja en la escuela (12m)',
    'ex11_15': 'Vio arma de fuego en la escuela (12m)',
    'ex12_16': 'Vio cuchillo/navaja en el barrio (12m)',
    'ex13_17': 'Vio arma de fuego en el barrio (12m)',
    'ex14_18': 'Conoce lugar para conseguir un arma de fuego',
    'ex15_19': 'Le ofrecieron un arma de fuego (12m)',
}
armas = []
for v, lab in labels_armas.items():
    p, n = pct_si(mod[v])
    armas.append({"var": v, "label": lab, "pct": round(p, 1), "n": n})

armas.sort(key=lambda r: r['pct'], reverse=True)
pd.DataFrame(armas)

## 10. Modelo de riesgo: víctima de robo dentro de la escuela

Variable dependiente: `pi5` (el ítem individual más prevalente de los 8 de
victimización). Predictores: sociodemográficos, consumo de sustancias,
entorno de pares, confianza institucional, y exposición a armas en la
escuela.

In [ ]:
d = df.copy()
d['y'] = np.where(d['pi5'] == SI, 1, np.where(d['pi5'] == NO, 0, np.nan))
d['sexo_m'] = np.where(d['d1'] == 'Hombre', 1, np.where(d['d1'] == 'Mujer', 0, np.nan))
d['edad'] = pd.to_numeric(d['d3'], errors='coerce')
d['supervision'] = d['p17'].map({'1. Mucho': 4, '2. Bastante': 3, '3. Poco': 2, '4. Nada': 1})
d['amigos_desaprueban'] = d['p76'].map({
    '1. Te harían algún reclamo o te dirían algo para que no lo hicieras': 1,
    '2. Algunos te harían reclamo y otros no': 0.5,
    '3. No te harían ningún reclamo o no te dirían nada': 0})
d['confia_policia'] = d['ci2'].map(MAP_CONF)
d['confia_escuela'] = d['ci1'].map(MAP_CONF)
for v in ['ma1', 'al1', 'ta1', 'co1']:
    d[f'{v}_d'] = np.where(d[v] == SI, 1, np.where(d[v] == NO, 0, np.nan))
d['arma_fuego_escuela'] = np.where(d['ex11_15'] == SI, 1, np.where(d['ex11_15'] == NO, 0, np.nan))
d['cuchillo_escuela'] = np.where(d['ex10_14'] == SI, 1, np.where(d['ex10_14'] == NO, 0, np.nan))

covars = ['sexo_m', 'edad', 'supervision', 'amigos_desaprueban', 'confia_policia', 'confia_escuela',
          'ma1_d', 'al1_d', 'ta1_d', 'co1_d', 'arma_fuego_escuela', 'cuchillo_escuela']
sub = d[['y'] + covars].dropna().copy()
print("N:", len(sub), " | Prevalencia:", round(sub['y'].mean() * 100, 1), "%")

In [ ]:
formula = ("y ~ sexo_m + edad + supervision + amigos_desaprueban + confia_policia + confia_escuela + "
           "ma1_d + al1_d + ta1_d + co1_d + arma_fuego_escuela + cuchillo_escuela")
model = smf.logit(formula, data=sub).fit(disp=0)
auc = roc_auc_score(sub['y'], model.predict(sub))
print(model.summary())
print("\nAUC:", round(auc, 3))

In [ ]:
labels_model = {'sexo_m': 'Ser hombre', 'edad': 'Edad (por año)', 'supervision': 'Supervisión parental',
                'amigos_desaprueban': 'Amigos desaprueban conducta de riesgo',
                'confia_policia': 'Confianza en la Policía', 'confia_escuela': 'Confianza en autoridades de la escuela',
                'ma1_d': 'Consumo de marihuana', 'al1_d': 'Consumo de alcohol', 'ta1_d': 'Consumo de tabaco',
                'co1_d': 'Consumo de cocaína', 'arma_fuego_escuela': 'Vio arma de fuego en la escuela',
                'cuchillo_escuela': 'Vio cuchillo/navaja en la escuela'}

factores = []
for var in model.params.index:
    if var == 'Intercept':
        continue
    factores.append({
        "var": var, "label": labels_model.get(var, var),
        "or": round(float(np.exp(model.params[var])), 2),
        "p": round(float(model.pvalues[var]), 4),
        "sig": bool(model.pvalues[var] < 0.05),
    })
factores.sort(key=lambda r: r['or'], reverse=True)

modelo_riesgo = {"n": int(len(sub)), "auc": round(float(auc), 2),
                 "prevalencia": round(float(sub['y'].mean()) * 100, 1), "factores": factores}

or_df = pd.DataFrame(factores)[['label', 'or', 'p', 'sig']]
or_df

## 11. Concentración territorial

Víctima de robo dentro de la escuela, por administración zonal, tipo de
sostenimiento y parroquia. No incluye escuelas FISCAL (no recibieron el
módulo).

In [ ]:
ZONA_MAP = {
    'CENTRO HISTORICO': 'MANUELA SÁENZ', 'SAN JUAN': 'MANUELA SÁENZ',
    'ITCHIMBIA': 'MANUELA SÁENZ', 'PUENGASI': 'MANUELA SÁENZ',
    'INAQUITO': 'EUGENIO ESPEJO', 'BELISARIO QUEVEDO': 'EUGENIO ESPEJO',
    'KENNEDY': 'EUGENIO ESPEJO', 'COCHAPAMBA': 'EUGENIO ESPEJO',
    'RUMIPAMBA': 'EUGENIO ESPEJO', 'LA CONCEPCION': 'EUGENIO ESPEJO',
    'JIPIJAPA': 'EUGENIO ESPEJO', 'MARISCAL SUCRE': 'EUGENIO ESPEJO',
    'SAN ISIDRO DEL INCA': 'EUGENIO ESPEJO',
    'LA MAGDALENA': 'ELOY ALFARO', 'SAN BARTOLO': 'ELOY ALFARO',
    'CHIMBACALLE': 'ELOY ALFARO', 'CHIMBCALLE': 'ELOY ALFARO',
    'LA FERROVIARIA': 'ELOY ALFARO', 'SOLANDA': 'ELOY ALFARO',
    'CHILIBULO': 'ELOY ALFARO',
    'CHILLOGALLO': 'QUITUMBE', 'QUITUMBE': 'QUITUMBE', 'TURUBAMBA': 'QUITUMBE',
    'GUAMANI': 'QUITUMBE', 'LA ECUATORIANA': 'QUITUMBE',
    'CALDERON (CARAPUNGO)': 'CALDERÓN', 'CALDERON': 'CALDERÓN',
    'POMASQUI': 'LA DELICIA', 'EL CONDADO': 'LA DELICIA', 'COTOCOLLAO': 'LA DELICIA',
    'PONCEANO': 'LA DELICIA', 'COMITE DEL PUEBLO': 'LA DELICIA',
    'SAN JOSE DE MINAS': 'LA DELICIA',
    'TUMBACO': 'TUMBACO', 'CUMBAYA': 'TUMBACO', 'GUAYLLABAMBA': 'TUMBACO',
    'EL QUINCHE': 'TUMBACO', 'CHECA': 'TUMBACO', 'PUEMBO': 'TUMBACO',
    'NAYON': 'TUMBACO',
    'PINTAG': 'LOS CHILLOS', 'CONOCOTO': 'LOS CHILLOS', 'ALANGASI': 'LOS CHILLOS',
    'AMAGUANA': 'LOS CHILLOS',
}

mod2 = mod.copy()
mod2['zona'] = mod2['e1'].map(ZONA_MAP)

zonas = []
for z, g in mod2.groupby('zona'):
    p, n = pct_si(g['pi5'])
    zonas.append({"zona": z, "pct": round(p, 1), "n": n})
zonas.sort(key=lambda r: r['pct'], reverse=True)

sostenimiento = []
for s, g in mod2.groupby('e2'):
    p, n = pct_si(g['pi5'])
    sostenimiento.append({"sostenimiento": s, "pct": round(p, 1), "n": n})
sostenimiento.sort(key=lambda r: r['pct'], reverse=True)

parroquias = []
for p_, g in mod2.groupby('e1'):
    pv, n = pct_si(g['pi5'])
    if n >= 100:
        parroquias.append({"parroquia": p_, "pct": round(pv, 1), "n": n})
parroquias.sort(key=lambda r: r['pct'], reverse=True)

territorial = {
    "nota": "No incluye escuelas FISCAL (16) ni 2 particulares: no recibieron el módulo 7.",
    "zonas": zonas, "sostenimiento": sostenimiento, "parroquias": parroquias,
}

print("--- Por zona ---")
display(pd.DataFrame(zonas))
print("--- Por sostenimiento ---")
display(pd.DataFrame(sostenimiento))
print("--- Top parroquias (n>=100) ---")
display(pd.DataFrame(parroquias).head(10))

## 12. Guardar `percepcion_data.json`

Este archivo es el insumo único para construir el HTML institucional
(`build_html_percepcion.py`) — ninguna cifra se escribe a mano en el resto
del pipeline.

In [ ]:
percepcion_data = {
    "conciliacion": conciliacion,
    "percepcion": percepcion,
    "victimizacion": victimizacion,
    "victimizacion_resumen": victimizacion_resumen,
    "confianza": confianza,
    "espacios": espacios,
    "armas": armas,
    "modelo_riesgo": modelo_riesgo,
    "territorial": territorial,
}

with open('percepcion_data.json', 'w', encoding='utf-8') as f:
    json.dump(percepcion_data, f, ensure_ascii=False, indent=2)

print("Guardado percepcion_data.json")
print(json.dumps(conciliacion, ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
files.download('percepcion_data.json')

---
**Siguiente paso**: usar este `percepcion_data.json` con
`build_html_percepcion.py` para regenerar el HTML institucional (línea
gráfica OMSCGR / Secretaría de Seguridad Ciudadana y Gestión de Riesgos),
sin volver a tocar la base cruda.